In [1]:
#https://www.wintick.com/members/symbolGuide/tda
import time
import pandas as pd
from IPython.display import display
from ipywidgets import widgets
#from ipywidgets import interact,interactive, interact_manual, Button, Output, HBox, VBox, Layout
#from threading import Thread
#import asyncio
#import os
#import glob
#import ast
#import json
#import csv
#import copy
import datetime
#import traceback
import functools
#import pytz

import ipyregulartable as rt
import ipysheet

#import globalvar
#from initialize import g_tick_dict, param, g_param, STRATEGY_LIST
#from trademanager import run_system, run_system_backtester


from ib_insync import *
from ACCOUNT_DETAIL import IP, PORT, CLIENT_ID
from IBtrader import get_account_detail, get_ticker
from IBtrader import place_order

In [2]:
util.startLoop()  # uncomment this line when in a notebook

global ibprogram
ibprogram = IB()


try:
    ibprogram.connect(IP, PORT, clientId=CLIENT_ID)
    #print(ibprogram)
    #acc = ibprogram.accountSummary()
    #ibprogram.sleep(2)
    
    #print("Connection Established!")
    accdata = get_account_detail(ibprogram)
    #print(accdata)
    dl=[{'AccountID':accdata['accountID'],'AvailableFunds':accdata['AvailableFunds'],'BuyingPower':accdata['BuyingPower']}]
    dfa=pd.DataFrame(dl)
except:
    print("Connection Failed? Refresh")
    dl = [{'account':0}]
    dfa  = pd.DataFrame(dl)    

<IB connected to 127.0.0.1:7497 clientId=999>
Connection Failed? Refresh


In [3]:
MAX_ROW = 3
MAX_COLUMN = 14
sheet = ipysheet.sheet(rows=MAX_ROW, columns=MAX_COLUMN)
#table_w = pd.DataFrame()#columns=['A','B','C','D','E','F','G','H','I','J','K','L','M','N'])
table_w = [[None for i in range(MAX_COLUMN)] for j in range(MAX_ROW)]

In [4]:
for i in range(MAX_ROW):
    if i==0:
        j=0
        table_w[i][j] = ipysheet.cell(i, j, 'Symbol',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'BID',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'ASK',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'MaxLoss',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'EntryPrice',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'StopPrice',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'Shares',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, '1stTarget',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, '2ndTarget',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'StopPct',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'Value',background_color='yellow')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'BUY',background_color='green')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'SELL',background_color='red')
        j+=1
        table_w[i][j] = ipysheet.cell(i, j, 'Comment',background_color='yellow')
    else:
        for jj in range(MAX_COLUMN):
            if jj==0:
                table_w[i][jj] = ipysheet.cell(i, jj, None,'text',background_color='white')
            elif jj==3:
                table_w[i][jj] = ipysheet.cell(i, jj, 0,'numeric',background_color='red')
            elif jj==4:
                table_w[i][jj] = ipysheet.cell(i, jj, 0,'numeric',background_color='green')
            elif jj==5:
                table_w[i][jj] = ipysheet.cell(i, jj, 0,'numeric',background_color='orange')    
            elif jj==6:
                table_w[i][jj] = ipysheet.cell(i, jj, 0,'numeric',read_only=True)
            else:    
                table_w[i][jj] = ipysheet.cell(i, jj, 0, 'numeric')

In [5]:
#table_w

In [6]:
from widgetlib import create_button,create_dropdown
param = {}
symbol_df = pd.read_csv('./SP500.csv')
symbol_list = symbol_df['Symbol'].to_list()

In [7]:
buyt = [None]*MAX_ROW
sellt = [None]*MAX_ROW
for i in range(1,MAX_ROW):
    buyt[i] = create_button(param, "BUY", 0, 'BUY','green')
    sellt[i] = create_button(param, "SELL", 0, 'SELL', 'red')
    ipysheet.cell(i, 11, buyt[i])
    ipysheet.cell(i, 12, sellt[i])

In [8]:
for i in range(1,MAX_ROW):
    ipysheet.cell(i, 3, 300) # MAX LOSS
    #ipysheet.cell(i, 3, 300) # MAX LOSS

In [9]:
#table_w.loc[1,9]  = 100
#ipysheet.cell(1, 9, b1)
#drp1 = create_dropdown(param, 'symbol_list', symbol_list, 'AAPL', tag='')
#ipysheet.cell(1, 0, drp1)

In [10]:
out_live_position = widgets.Output()

In [11]:
#out_live_position.clear_output(False)
#with out_live_position:
#    display(sheet)       

In [12]:
#ipysheet.cell(1, 6) = sheet.cell(1, 4) - sheet.cell(1, 5)
def calculate_share(change, i=0):
    try:
        table_w[i][6].value = abs(int(table_w[i][3].value/(table_w[i][4].value - table_w[i][5].value)))
    except:
        table_w[i][6].value = 0
#table_w.loc[1,4].observe(calculate, 'value')
#table_w.loc[1,5].observe(calculate, 'value')
for r in range(1,MAX_ROW): 
    table_w[r][4].observe(functools.partial(calculate_share, i = r))
    table_w[r][5].observe(functools.partial(calculate_share, i = r))

In [13]:
def calculate_target1(change, i=0):
    try:
        table_w[i][7].value = table_w[i][4].value + (table_w[i][4].value - table_w[i][5].value)
    except:
        table_w[i][7].value = 0.0
for r in range(1,MAX_ROW): 
    table_w[r][4].observe(functools.partial(calculate_target1, i = r))
    table_w[r][5].observe(functools.partial(calculate_target1, i = r))

In [14]:
def calculate_target2(change, i=0):
    try:
        table_w[i][8].value = table_w[i][4].value + 2*(table_w[i][4].value - table_w[i][5].value)
    except:
        table_w[i][8].value = 0.0
#table_w.loc[1,4].observe(calculate, 'value')
#table_w.loc[1,5].observe(calculate, 'value')
for r in range(1,MAX_ROW): 
    table_w[r][4].observe(functools.partial(calculate_target2, i = r))
    table_w[r][5].observe(functools.partial(calculate_target2, i = r))

In [15]:
#ipysheet.cell(1, 6) = sheet.cell(1, 4) - sheet.cell(1, 5)
def calculate_stoppct(change, i=0):
    try:
        table_w[i][9].value = 100*(table_w[i][4].value - table_w[i][5].value)/table_w[i][4].value
    except:
        table_w[i][9].value = 0
#table_w[1,4].observe(calculate, 'value')
#table_w[1,5].observe(calculate, 'value')
for r in range(1,MAX_ROW): 
    table_w[r][4].observe(functools.partial(calculate_stoppct, i = r))
    table_w[r][5].observe(functools.partial(calculate_stoppct, i = r))

In [16]:
#ipysheet.cell(1, 6) = sheet.cell(1, 4) - sheet.cell(1, 5)
def calculate_value(change, i=0):
    try:
        table_w[i][10].value = (table_w[i][4].value * table_w[i][6].value)
    except:
        table_w[i][10].value = 0
#table_w[1,4].observe(calculate, 'value')
#table_w[1,5].observe(calculate, 'value')
for r in range(1,MAX_ROW): 
    table_w[r][4].observe(functools.partial(calculate_value, i = r))
    table_w[r][5].observe(functools.partial(calculate_value, i = r))

In [20]:
def buy_button(b, i=0):
    try:
        symbol = table_w[i][0].value
        price = table_w[i][4].value
        sl = table_w[i][5].value
        
        size = table_w[i][6].value
        size1 = int(size/2)
        size2 = int(size - size1)
        
        tp1 = table_w[i][7].value
        tp2 = table_w[i][8].value
        print("BUY1:",symbol,"price:",price,"sl:",sl,"tp:",tp1,"size:",size1)
        place_order(instrument_type='stock',symbol =symbol,orderType='LMT',
                    direction="Buy", size=size1, price=price, TP=tp1, SL=sl,ibprogram=ibprogram)
        print("BUY2:",symbol,"price:",price,"sl:",sl,"tp:",tp2,"size:",size2)
        place_order(instrument_type='stock',symbol =symbol,orderType='LMT',
                    direction="Buy", size=size2, price=price, TP=tp2, SL=sl,ibprogram=ibprogram)
    except:
        print("buy failed for ", table_w[i][0].value)

def sell_button(b, i=0):
    try:
        symbol = table_w[i][0].value
        price = table_w[i][4].value
        sl = table_w[i][5].value

        size = table_w[i][6].value
        size1 = int(size/2)
        size2 = int(size - size1)

        tp1 = table_w[i][7].value
        tp2 = table_w[i][8].value
        print("SELL1:",symbol,"price:",price,"sl:",sl,"tp:",tp1,"size:",size1)
        place_order(instrument_type='stock',symbol =symbol,orderType='LMT',
                    direction="Sell", size=size1, price=price, TP=tp1, SL=sl,ibprogram=ibprogram)
        print("SELL2:",symbol,"price:",price,"sl:",sl,"tp:",tp2,"size:",size2)
        place_order(instrument_type='stock',symbol =symbol,orderType='LMT',
                    direction="Sell", size=size2, price=price, TP=tp2, SL=sl,ibprogram=ibprogram)
        #print("SELL:",symbol,"price:",price,"sl:",sl,"tp1:",tp1,"tp2:",tp2)
    except:
        print("sell failed for ", table_w[i][0].value)

In [18]:
#ipysheet.cell(1, 6) = sheet.cell(1, 4) - sheet.cell(1, 5)
ticker=[None]*30
def calculate_ticker(change, i=0):
    #try:
    return
    print(change)
    ticker[i] = get_ticker(ibprogram, change)
    print(ticker[i])
    table_w[i][1].value = ticker[i].bid  
    #except:
    #    table_w.loc[i,10].value = 0
for r in range(1,MAX_ROW): 
    table_w[r][0].observe(functools.partial(calculate_ticker, i = r))

In [21]:
for r in range(1,MAX_ROW):
    print(table_w[r][11].value)
    table_w[r][11].value.on_click(functools.partial(buy_button, i = r))
    table_w[r][12].value.on_click(functools.partial(sell_button, i = r))

Button(description='BUY', layout=Layout(width='90%'), style=ButtonStyle(button_color='green'))
Button(description='BUY', layout=Layout(width='90%'), style=ButtonStyle(button_color='green'))


In [ ]:
#from IBtrader import get_ticker
#ticker=[None]*30
#table_w.loc[1,0] = 'AAPL'
#ticker[0] = get_ticker(ibprogram, 'AAPL')
#print(ticker[0])
#table_w.loc[1,1].value = ticker[0].bid

In [22]:
#display(table_w)
sheet
#out_live_position

Sheet(cells=(Cell(column_end=0, column_start=0, row_end=0, row_start=0, style={'backgroundColor': 'yellow'}, t…

In [ ]:
#rows, cols = (30, 15)
#arr = [[None for i in range(cols)] for j in range(rows)]
#print(arr)

In [ ]:
#arr[1][2]=3
#print(arr)

In [ ]:
#from IBtrader import place_order
i=2

symbol = table_w[i][0].value
price = (table_w[i][4].value)
sl = (table_w[i][5].value)
tp1 = (table_w[i][7].value)
tp2 = (table_w[i][8].value)

print("BUY TYPE:",type(symbol),"price:",type(price),"sl:",type(sl),"tp1:",type(tp1),"tp2:",type(tp2))
print("BUY:",symbol,"price:",price,"sl:",sl,"tp1:",tp1,"tp2:",tp2)
#place_order(symbol=symbol,orderType='LMT',direction="Sell", size=1, price=price, TP=tp1, SL=sl,ibprogram=ibprogram)